In [3]:
import os
# Fixer le répertoire de travail sur la racine de ton projet
os.chdir('/Users/7maksacod/Documents/ISM/Master_2/Memoire_big_data_ia')

In [5]:
import matplotlib
matplotlib.use('Agg')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import FuncFormatter

# 0. Configuration du style graphique
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'font.size': 12, 'figure.titlesize': 16, 'axes.titlesize': 14})

print("📥 Chargement et préparation du dataset Gold pour l'EDA...")
df = pd.read_csv('data/processed/dataset_dakar_gold.csv', sep=';')

# EXCLUSION DES OUTLIERS (Salles de bain > 10)
df = df[df['nb_salles_bain'] <= 10].copy()
print(f"📊 Volume analysé : {len(df)} annonces x {df.shape[1]} variables.\n")

def millions_formatter(x, pos):
    return f"{x*1e-6:.1f}M"

# ==============================================================================
# 1. STATISTIQUES DÉTAILLÉES DE LA VARIABLE CIBLE (PRIX)
# ==============================================================================
print("==================================================================")
print("--- 1. ANALYSE DÉTAILLÉE DE LA VARIABLE CIBLE (prix_fcfa) ---")
print("==================================================================")
stats_prix = df['prix_fcfa'].describe()
q25, q75 = stats_prix['25%'], stats_prix['75%']
iqr = q75 - q25

print(f"• Prix Minimum     : {stats_prix['min']:,.0f} FCFA")
print(f"• Prix Médian (50%): {stats_prix['50%']:,.0f} FCFA")
print(f"• Prix Moyen       : {stats_prix['mean']:,.0f} FCFA")
print(f"• Prix Maximum     : {stats_prix['max']:,.0f} FCFA")
print(f"• Écart-type (Std) : {stats_prix['std']:,.0f} FCFA")
print(f"• Écart Interquartile (IQR) : {iqr:,.0f} FCFA [25%: {q25:,.0f} | 75%: {q75:,.0f}]")
print(f"• Coefficient d'Asymétrie (Skewness) : {df['prix_fcfa'].skew():.2f} (Très étiré vers la droite)")
print(f"• Coefficient d'Aplatissement (Kurtosis) : {df['prix_fcfa'].kurtosis():.2f}\n")

# Figure 1 : Distribution Brute vs Logarithmique
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.histplot(df['prix_fcfa'], kde=True, bins=40, color='#e74c3c', ax=axes[0])
axes[0].set_title('A. Distribution des Loyers (Brut)')
axes[0].set_xlabel('Loyer Mensuel (FCFA)')
axes[0].set_ylabel('Nombre d\'annonces')
axes[0].xaxis.set_major_formatter(FuncFormatter(millions_formatter))

sns.histplot(np.log1p(df['prix_fcfa']), kde=True, bins=40, color='#2ecc71', ax=axes[1])
axes[1].set_title('B. Distribution Logarithmique (np.log1p)')
axes[1].set_xlabel('Log(Loyer Mensuel)')
axes[1].set_ylabel('Nombre d\'annonces')

plt.suptitle('Figure 1 : Normalisation de la Variable Cible pour le Machine Learning', y=1.02)
plt.tight_layout()
plt.savefig('eda_1_price_distribution_dual.png', dpi=300, bbox_inches='tight')
plt.close(fig)

# ==============================================================================
# 2. PANORAMA DES ÉQUIPEMENTS & SURCOTE DE STANDING
# ==============================================================================
print("==================================================================")
print("--- 2. TAUX DE PÉNÉTRATION & IMPACT DES ÉQUIPEMENTS SUR LE PRIX ---")
print("==================================================================")
commodites = {
    'Vue sur Mer': 'has_vue_mer', 'Piscine': 'has_piscine', 
    'Sécurité H24': 'has_securite', 'Parking/Garage': 'has_parking',
    'Groupe Électrogène': 'has_groupe', 'Ascenseur': 'has_ascenseur',
    'Meublé': 'is_meuble', 'Climatisation': 'has_clim', 'Cuisine Équipée': 'has_cuisine_equipee'
}

impact_list = []
for nom, col in commodites.items():
    taux_pen = (df[col].mean() * 100)
    med_avec = df[df[col] == 1]['prix_fcfa'].median()
    med_sans = df[df[col] == 0]['prix_fcfa'].median()
    if pd.notna(med_avec) and pd.notna(med_sans) and med_sans > 0:
        prime_pct = ((med_avec - med_sans) / med_sans) * 100
        prime_val = med_avec - med_sans
        impact_list.append({
            'Équipement': nom,
            'Pénétration (%)': round(taux_pen, 1),
            'Médiane Avec': f"{med_avec:,.0f} F",
            'Médiane Sans': f"{med_sans:,.0f} F",
            'Surcote (FCFA)': f"+{prime_val:,.0f} F",
            'Prime (%)': round(prime_pct, 1)
        })

df_impact = pd.DataFrame(impact_list).sort_values(by='Prime (%)', ascending=False)
print(df_impact.to_string(index=False))
print("\n")

# Figure 4 : Graphique de la Prime d'équipement
df_impact_graph = df_impact.sort_values(by='Prime (%)', ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(df_impact_graph['Équipement'], df_impact_graph['Prime (%)'], color='#9b59b6')
ax.set_title('Figure 4 : Surcote Tarifaire Apportée par les Équipements (%)')
ax.set_xlabel('Augmentation de la valeur médiane du loyer (%)')

for bar in bars:
    width = bar.get_width()
    ax.text(width + 3, bar.get_y() + bar.get_height()/2, f'+{int(width)}%', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('eda_4_amenities_premium.png', dpi=300)
plt.close(fig)

# ==============================================================================
# 3. TYPOLOGIE DES BIENS & QUARTIERS
# ==============================================================================
print("==================================================================")
print("--- 3. HIÉRARCHIE DES QUARTIERS ET TYPES DE BIENS ---")
print("==================================================================")
# Exclusion explicite de "Inconnu" et filtre sur volume significatif (>= 20)
quartiers_significatifs = df[df['quartier'] != 'Inconnu']['quartier'].value_counts()[df[df['quartier'] != 'Inconnu']['quartier'].value_counts() >= 20].index
df_q = df[df['quartier'].isin(quartiers_significatifs)]

print("• Loyers Médian par Quartier Principal :")
q_summary = df_q.groupby('quartier')['prix_fcfa'].agg(['count', 'median', 'mean']).sort_values(by='median', ascending=False)
q_summary.columns = ['Nb Annonces', 'Loyer Médian (FCFA)', 'Loyer Moyen (FCFA)']
print(q_summary.to_string())
print("\n")

# Figure 2 : Boxplot des quartiers
ordre_quartiers = df_q.groupby('quartier')['prix_fcfa'].median().sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(14, 7))
sns.boxplot(data=df_q, x='quartier', y='prix_fcfa', order=ordre_quartiers, palette='viridis', ax=ax)
ax.set_title('Figure 2 : Hiérarchie des Loyers par Quartier (Trié par Médiane)')
ax.set_xlabel('Quartier')
ax.set_ylabel('Loyer Mensuel (FCFA)')
ax.yaxis.set_major_formatter(FuncFormatter(millions_formatter))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('eda_2_boxplot_quartiers_tries.png', dpi=300, bbox_inches='tight')
plt.close(fig)

# ==============================================================================
# 4. RELATION AVEC LA DISTANCE ET LES PIÈCES
# ==============================================================================
print("==================================================================")
print("--- 4. MATRICE DE CORRÉLATION ET DYNAMIQUE SPATIALE ---")
print("==================================================================")
# Figure 3 : Distance au Plateau
fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=df, x='dist_plateau_km', y=np.log1p(df['prix_fcfa']), 
            scatter_kws={'alpha':0.3, 'color': '#3498db'}, line_kws={'color': '#e74c3c', 'lw': 3}, ax=ax)
ax.set_title('Figure 3 : Décroissance du Loyer selon l\'Éloignement du Plateau')
ax.set_xlabel('Distance au Plateau (km)')
ax.set_ylabel('Log(Loyer Mensuel)')
plt.tight_layout()
plt.savefig('eda_3_distance_gradient.png', dpi=300)
plt.close(fig)

# Figure 5 : Matrice de Corrélation
corr_cols = ['prix_fcfa', 'nb_pieces', 'nb_salles_bain', 
             'dist_plateau_km', 'dist_vdn_km', 'dist_mer_km', 
             'has_piscine', 'has_parking', 'has_ascenseur', 'is_meuble']

corr_matrix = df[corr_cols].corr(method='spearman')
print("• Matrice de Corrélation de Spearman (par rapport au Prix FCFA) :")
print(corr_matrix['prix_fcfa'].sort_values(ascending=False).to_string())
print("\n")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm', center=0, vmin=-0.5, vmax=1.0, 
            square=True, linewidths=.5, cbar_kws={"shrink": .8}, ax=ax)
ax.set_title('Figure 5 : Matrice de Corrélation de Spearman')
plt.tight_layout()
plt.savefig('eda_5_correlation_matrix.png', dpi=300)
plt.close(fig)

print("🚀 ANALYSE EXPLORATOIRE COMPLÈTE ET EXÉCUTÉE AVEC SUCCÈS !")

📥 Chargement et préparation du dataset Gold pour l'EDA...
📊 Volume analysé : 3515 annonces x 31 variables.

--- 1. ANALYSE DÉTAILLÉE DE LA VARIABLE CIBLE (prix_fcfa) ---
• Prix Minimum     : 30,000 FCFA
• Prix Médian (50%): 400,000 FCFA
• Prix Moyen       : 559,484 FCFA
• Prix Maximum     : 3,500,000 FCFA
• Écart-type (Std) : 522,301 FCFA
• Écart Interquartile (IQR) : 462,500 FCFA [25%: 237,500 | 75%: 700,000]
• Coefficient d'Asymétrie (Skewness) : 2.18 (Très étiré vers la droite)
• Coefficient d'Aplatissement (Kurtosis) : 5.95

--- 2. TAUX DE PÉNÉTRATION & IMPACT DES ÉQUIPEMENTS SUR LE PRIX ---
        Équipement  Pénétration (%) Médiane Avec Médiane Sans Surcote (FCFA)  Prime (%)
       Vue sur Mer              4.4  1,500,000 F    375,000 F   +1,125,000 F      300.0
           Piscine              9.6  1,200,000 F    350,000 F     +850,000 F      242.9
    Parking/Garage             20.1  1,000,000 F    350,000 F     +650,000 F      185.7
Groupe Électrogène             12.6    850,00

/var/folders/_v/q433ftnj7s92pxmjlmtwxbq80000gn/T/ipykernel_62619/3154684268.py:129: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_q, x='quartier', y='prix_fcfa', order=ordre_quartiers, palette='viridis', ax=ax)


--- 4. MATRICE DE CORRÉLATION ET DYNAMIQUE SPATIALE ---
• Matrice de Corrélation de Spearman (par rapport au Prix FCFA) :
prix_fcfa          1.000000
nb_pieces          0.659007
has_parking        0.522236
nb_salles_bain     0.502867
is_meuble          0.434690
has_piscine        0.400127
has_ascenseur      0.386078
dist_vdn_km        0.017300
dist_plateau_km   -0.031121
dist_mer_km       -0.263716


🚀 ANALYSE EXPLORATOIRE COMPLÈTE ET EXÉCUTÉE AVEC SUCCÈS !
